# 4 · Query the Supervisor Agent

Queries the Supervisor Agent endpoint created in notebook `3`. The supervisor
routes each question to the right sub-agent (contracts retriever or
sales analytics Genie space) and returns a unified answer.

### Install Dependencies

In [0]:
%pip install --upgrade databricks-sdk openai
dbutils.library.restartPython()

### Define Parameters

In [0]:
dbutils.widgets.text("SUPERVISOR_NAME", "Marketing Campaign Insights Supervisor", "Supervisor Agent Name")
dbutils.widgets.text(
    "QUESTION",
    "Which suppliers do we have contracts with, and how do their agreed terms compare?",
    "Question",
)

SUPERVISOR_NAME = dbutils.widgets.get("SUPERVISOR_NAME")
QUESTION = dbutils.widgets.get("QUESTION")

### Resolve the Endpoint Name

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

sup = next(
    (s for s in w.supervisor_agents.list_supervisor_agents()
     if s.display_name == SUPERVISOR_NAME),
    None,
)
assert sup is not None, f"Supervisor Agent '{SUPERVISOR_NAME}' not found. Run notebook 3 first."

ENDPOINT_NAME = sup.endpoint_name
print("Endpoint:", ENDPOINT_NAME)

### Query the Endpoint

In [0]:
client = w.serving_endpoints.get_open_ai_client()

def ask(question: str) -> str:
    try:
        resp = client.responses.create(
            model=ENDPOINT_NAME,
            input=[{"role": "user", "content": question}],
        )
        return resp.output_text
    except Exception:
        resp = client.chat.completions.create(
            model=ENDPOINT_NAME,
            messages=[{"role": "user", "content": question}],
        )
        return resp.choices[0].message.content

answer = ask(QUESTION)
print("Q:", QUESTION)
print("\nA:", answer)

### Try a Few More Routing Examples

In [0]:
examples = [
    "What are the total sales by store type?",              # -> Genie
    "Summarize the termination clauses across the contracts.",  # -> contracts retriever
]
for q in examples:
    print("=" * 80)
    print("Q:", q)
    print("A:", ask(q))